In [1]:
import math
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # km

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c * 1000 # km

In [2]:
import pandas as pd

df = pd.read_csv("BWSC route_csv/Route1.csv")
df

,ele,lat,lon,name
0,38.688,-12.466018,130.842784,Route
1,38.688,-12.466018,130.842784,Route
2,36.326,-12.465811,130.842551,Route
3,36.905,-12.465770,130.842519,Route
4,37.471,-12.465732,130.842498,Route
...,...,...,...,...
5452,113.059,-14.481378,132.324323,Route
5453,113.113,-14.481399,132.324477,Route
5454,113.626,-14.481453,132.324670,Route
5455,113.440,-14.481545,132.325030,Route


In [3]:
k = haversine(df.iloc[0, 1], df.iloc[0, 2], df.iloc[3, 1], df.iloc[3, 2])
print(f"{k:.03f}")

39.846


In [4]:
import os

_, _, file_list = next(os.walk("BWSC route_csv"))
for idx, _ in enumerate(file_list):
    if file_list[idx] == "80kmRoute.csv":
        del file_list[idx]
file_list.sort(key=lambda x: int(x[5:-4]))
file_list

['Route1.csv',
 'Route2.csv',
 'Route3.csv',
 'Route4.csv',
 'Route5.csv',
 'Route6.csv',
 'Route7.csv',
 'Route8.csv',
 'Route9.csv',
 'Route10.csv']

In [5]:
import pandas as pd

Route = "BWSC route_csv"
df = pd.DataFrame()
for file in file_list:
    df = pd.concat([df, pd.read_csv(Route+"/"+file)], axis=0)

df

,ele,lat,lon,name
0,38.688,-12.466018,130.842784,Route
1,38.688,-12.466018,130.842784,Route
2,36.326,-12.465811,130.842551,Route
3,36.905,-12.465770,130.842519,Route
4,37.471,-12.465732,130.842498,Route
...,...,...,...,...
4678,58.158,-34.927596,138.621069,Route10
4679,57.578,-34.927588,138.620891,Route10
4680,57.991,-34.927589,138.620574,Route10
4681,55.831,-34.927662,138.619010,Route10


In [6]:
df.index = [x for x in range(38210)]
df.index

Index([    0,     1,     2,     3,     4,     5,     6,     7,     8,     9,
       ...
       38200, 38201, 38202, 38203, 38204, 38205, 38206, 38207, 38208, 38209],
      dtype='int64', length=38210)

In [7]:
for index in df.index:
    if index==0:
        df['distance'] = 0.0
        continue
    
    df.loc[index, 'distance'] = haversine(df.loc[index-1, 'lat'], df.loc[index-1, 'lon'], df.loc[index, 'lat'], df.loc[index, 'lon'])
    df.loc[index, 'distance'] += df.loc[index-1, 'distance']

df

,ele,lat,lon,name,distance
0,38.688,-12.466018,130.842784,Route,0.000000e+00
1,38.688,-12.466018,130.842784,Route,0.000000e+00
2,36.326,-12.465811,130.842551,Route,3.428639e+01
3,36.905,-12.465770,130.842519,Route,3.992132e+01
4,37.471,-12.465732,130.842498,Route,4.482584e+01
...,...,...,...,...,...
38205,58.158,-34.927596,138.621069,Route10,3.038133e+06
38206,57.578,-34.927588,138.620891,Route10,3.038150e+06
38207,57.991,-34.927589,138.620574,Route10,3.038179e+06
38208,55.831,-34.927662,138.619010,Route10,3.038321e+06


In [11]:
targets = np.arange(0, df['distance'].max(), 80)

indices = []

for t in targets:
    idx = (df['distance'] - t).abs().idxmin()
    indices.append(idx)

df_80m = df.loc[indices].drop_duplicates()
df_80m

,ele,lat,lon,name,distance
0,38.688,-12.466018,130.842784,Route,0.000000e+00
10,40.116,-12.465523,130.842627,Route,8.069656e+01
16,39.141,-12.464773,130.843397,Route,1.991709e+02
18,37.385,-12.464375,130.843787,Route,2.604598e+02
21,37.291,-12.464066,130.844093,Route,3.082495e+02
...,...,...,...,...,...
38195,58.289,-34.927710,138.622494,Route10,3.038001e+06
38201,59.254,-34.927642,138.621605,Route10,3.038084e+06
38206,57.578,-34.927588,138.620891,Route10,3.038150e+06
38207,57.991,-34.927589,138.620574,Route10,3.038179e+06


In [12]:
df_80m.drop("name",axis=1, inplace=True)

In [13]:
df_80m.to_csv("80mRoute.csv")

In [20]:
se = df[['distance']] // 80
se = se.drop_duplicates(subset='distance', keep='first')
se

,distance
0,0.0
10,1.0
16,2.0
18,3.0
22,4.0
...,...
38118,37974.0
38195,37975.0
38201,37976.0
38207,37977.0


In [34]:
df_selected = df.loc[se.index]
df_selected['temp'] = se['distance']
df_selected

,ele,lat,lon,name,distance,temp
0,38.688,-12.466018,130.842784,Route,0.000000e+00,0.0
10,40.116,-12.465523,130.842627,Route,8.069656e+01,1.0
16,39.141,-12.464773,130.843397,Route,1.991709e+02,2.0
18,37.385,-12.464375,130.843787,Route,2.604598e+02,3.0
22,36.143,-12.463871,130.844295,Route,3.391211e+02,4.0
...,...,...,...,...,...,...
38118,58.375,-34.927461,138.622595,Route10,3.037944e+06,37974.0
38195,58.289,-34.927710,138.622494,Route10,3.038001e+06,37975.0
38201,59.254,-34.927642,138.621605,Route10,3.038084e+06,37976.0
38207,57.991,-34.927589,138.620574,Route10,3.038179e+06,37977.0


In [35]:
df_selected.drop('name', inplace=True, axis=1)
df_selected.to_csv('80mRoute.csv', index=True)